In [ ]:
# test_incremental.py
import pickle, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

BASE = "/mnt/volume6/czj/labLGN/LabLZ/data_preparation/"
LAB  = pd.read_csv(BASE + "cell2024_model_single_subst.csv")
LAB["KEY"] = LAB["Gene"].astype(str) + "||" + LAB["mutation_used"].astype(str)
esm = pickle.load(open(BASE + "phase4_esm2_local_delta.pkl", "rb"))                 # KEY->(1280,)
pup = pickle.load(open("/mnt/volume6/czj/labLGN/LabLZ/pups_trial/pups_full_delta.pkl", "rb"))  # KEY->(29,)

# align three datasets
rows = []
for _, r in LAB.iterrows():
    k = r["KEY"]
    if pd.isna(r["Mislocalized"]):
        continue
    e, p = esm.get(k), pup.get(k)
    if e is None or p is None:
        continue
    rows.append((r["Gene"], int(r["Mislocalized"]), e, p))

genes = np.array([r[0] for r in rows])
y     = np.array([r[1] for r in rows])
E     = np.vstack([r[2] for r in rows])   # ESM2 local delta  (n, 1280)
P     = np.vstack([r[3] for r in rows])   # PUPS delta        (n, 29)

print(f"Sample n={len(y)}, Positive={int(y.sum())}, Genes={len(set(genes))}")

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
def auc(X, name):
    pipe = make_pipeline(SimpleImputer(), StandardScaler(),
                         LogisticRegression(max_iter=2000, class_weight="balanced"))
    oof = cross_val_predict(pipe, X, y, cv=cv, groups=genes, method="predict_proba")[:, 1]
    a = roc_auc_score(y, oof); print(f"  {name:26s} AUROC={a:.3f}"); return a

print("=== Incremental test ===")
auc(P, "PUPS only")
b = auc(E, "ESM2-delta only")
c = auc(np.hstack([E, P]), "ESM2-delta + PUPS")
print(f"\n Δ = {c - b:+.3f} → {'PUPS is good' if c - b > 0.02 else 'PUPS is bad'}")

Sample n=2179, Positive=236, Genes=871
=== Incremental test ===
  PUPS only                  AUROC=0.526
  ESM2-delta only            AUROC=0.516
  ESM2-delta + PUPS          AUROC=0.526

 Δ = +0.010 → PUPS is bad
